<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Your first AI model: will this ticket escalate?

Companion notebook for the video *Train your first AI model in 10 minutes, nothing to install* · from the free course [Machine Learning: From Problem to Reliable Model](https://learning.nextia-ai.com/courses/ml/) · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** load real-looking support tickets, build a lazy baseline, train a simple model, check it honestly, see a common mistake, and predict a new ticket.

| | |
|---|---|
| **Time** | About 10 minutes |
| **Needs** | A browser. No account for the data. The first cell installs pandas and scikit-learn. |
| **Data** | Larkfield, a made-up Canadian online shop for home and garden products. The tickets are synthetic: there are no real people. |

Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.

## Step 1. Install the packages

This cell installs pandas (tables) and scikit-learn (machine learning). It takes up to a minute the first time.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1
import pandas as pd, sklearn
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## Step 2. Load the Larkfield tickets

One row is one support ticket. It has only the facts that are known **when the ticket is created**, and one answer: `escalated_72h` is 1 when the ticket was passed to a senior agent within 72 hours, and 0 when it was not.

In [ ]:
import hashlib, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C05/data/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
}
Path("larkfield").mkdir(exist_ok=True)
for name, expected in FILES.items():
    path = Path("larkfield") / name
    if not path.exists():
        urllib.request.urlretrieve(DATA + name, path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f"{name} is damaged: delete it and run again"

tickets = pd.concat([pd.read_csv(f"larkfield/{n}") for n in ["train.csv", "valid.csv", "test.csv"]], ignore_index=True)
print(f"Loaded {len(tickets):,} tickets.")

## Step 3. Look at the data first

Before we train anything, we look: the columns, a few rows, and the share of tickets that escalated.

In [ ]:
tickets.head()

In [ ]:
print("Columns:", ", ".join(tickets.columns))
escalated = tickets["escalated_72h"].sum()
print(f"{escalated:,} of {len(tickets):,} tickets escalated: {escalated / len(tickets):.1%}")

## Step 4. Split: learn from the past, test on a later month

The model learns from the older tickets, and we test it on the June 2026 tickets, which it never sees during training. This is like a teacher who keeps the exam questions out of the practice sheets.

In [ ]:
learn = tickets[tickets["created_at"] < "2026-06-01"]
test = tickets[tickets["created_at"] >= "2026-06-01"]
risky = test["escalated_72h"].sum()
print(f"Learn from: {len(learn):,} tickets (July 2024 to May 2026)")
print(f"Test on:    {len(test):,} tickets (June 2026), of which {risky} escalated")

## Step 5. The lazy baseline

A lazy "model" that always says **no escalation**. Every real model must beat it on the measure that matters: how many risky tickets we catch.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

truth = test["escalated_72h"]
lazy = [0] * len(test)
print(f"Lazy baseline accuracy: {accuracy_score(truth, lazy):.1%}")
print(f"Risky tickets caught:   {recall_score(truth, lazy):.0%}  (0 of {risky})")

## Step 6. Train a simple model

The model is a logistic regression: it gives each ticket a **risk score** between 0 and 1. It uses only facts known when the ticket arrives: the team, the channel, the priority, the type of customer, the region, the order value, the length of the first message, how long the customer has been with Larkfield, their tickets in the last 90 days, and the hour.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d", "created_hour"]

def make_model(numbers):
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numbers),
    ])
    return make_pipeline(prepare, LogisticRegression(max_iter=2000))

model = make_model(NUMBERS).fit(learn, learn["escalated_72h"])
print("Trained.")

## Step 7. Did it work? Compare with the baseline

The team has time to check about one ticket in five first. So the model **flags** a ticket when its risk score is 0.20 or more, which flags about one ticket in five.

- **Caught** (recall): of the risky tickets, how many did we flag?
- **Right when it flags** (precision): of the flagged tickets, how many really escalated? The rest are false alarms.

In [ ]:
CUT_OFF = 0.20
risk = model.predict_proba(test)[:, 1]
flag = (risk >= CUT_OFF).astype(int)

def report(name, predicted):
    caught = int(((truth == 1) & (pd.Series(predicted, index=truth.index) == 1)).sum())
    flagged = int(sum(predicted))
    print(f"{name:15} accuracy {accuracy_score(truth, predicted):.1%} | caught {caught} of {risky} ({recall_score(truth, predicted):.0%})"
          f" | flagged {flagged}, right {precision_score(truth, predicted, zero_division=0):.0%}, false alarms {flagged - caught}")

report("Lazy baseline", lazy)
report("Model", flag)

## Step 8. A mistake, on purpose: a column from the future

The extra file has `priority_now`: the ticket's priority **today**. When a ticket escalates, a senior agent sets it to 1. So this column is filled after the answer is known. Watch the score.

In [ ]:
extra = pd.read_csv("larkfield/extra_features.csv")[["ticket_id", "priority_now"]]
learn_x = learn.merge(extra, on="ticket_id")
test_x = test.merge(extra, on="ticket_id")

leaky = make_model(NUMBERS + ["priority_now"]).fit(learn_x, learn_x["escalated_72h"])
leaky_flag = (leaky.predict_proba(test_x)[:, 1] >= CUT_OFF).astype(int)
report("Model", flag)
report("Leaky model", leaky_flag)

Almost perfect, and wrong. When a new ticket arrives, nobody knows `priority_now` yet, so this model cannot work in real life. Always ask: *could the model know this at the moment it makes the prediction?* We keep the honest model.

## Step 9. Predict a new ticket

A new ticket arrives. The model does not read the words: it uses the facts of the ticket. Change the facts, run the cell again, and watch the risk change.

In [ ]:
message = ("You charged our company card twice for the same patio furniture order. "
           "This is my third message this month and nobody has answered. Please refund the extra charge today.")
new_ticket = pd.DataFrame([{
    "channel": "email", "team": "payment", "priority": 1, "segment": "business", "region": "west",
    "order_value": 1240.0, "word_count": len(message.split()), "customer_tenure_days": 610,
    "prior_tickets_90d": 2, "created_hour": 9,
}])
p = model.predict_proba(new_ticket)[0, 1]
print(message)
print(f"Likely to escalate: {'YES' if p >= CUT_OFF else 'NO'} ({p:.2f})")

In [ ]:
# Your turn: a different ticket. Try other teams, priorities or customers.
message = "My parcel is two weeks late and nobody answers."
other = new_ticket.assign(team="delivery", priority=2, segment="home", region="east",
                          order_value=89.0, word_count=len(message.split()), customer_tenure_days=400, prior_tickets_90d=1, created_hour=10)
p = model.predict_proba(other)[0, 1]
print(message)
print(f"Likely to escalate: {'YES' if p >= CUT_OFF else 'NO'} ({p:.2f})")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## What you just did

You defined a question, looked at the data, built a lazy baseline, trained a model, tested it on a later month, and caught a leak. The free course [Machine Learning: From Problem to Reliable Model](https://learning.nextia-ai.com/courses/ml/) teaches each step properly, starting with [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). New to Python? Start with [Python for Practical AI Engineering](https://learning.nextia-ai.com/courses/python/).

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.